# SVM, K-Nearest Neighbors, and Naive Bayes

> Compare classic algorithms and understand their assumptions and scaling sensitivity.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Compare a margin-based model, a distance-based model, and a probabilistic baseline.
- Understand why scaling is important for SVM and KNN.
- Learn common use cases and limitations.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, accuracy_score

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
models = {
 "Scaled RBF SVM": make_pipeline(StandardScaler(), SVC(C=1.0, kernel="rbf", probability=True, random_state=42)),
 "Scaled KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
 "Gaussian Naive Bayes": GaussianNB(),
}
rows=[]
for name, est in models.items():
    est.fit(X_train, y_train)
    prob=est.predict_proba(X_test)[:,1]
    rows.append({"model":name, "accuracy":accuracy_score(y_test,est.predict(X_test)), "ROC-AUC":roc_auc_score(y_test,prob)})
display(pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).round(3))

,model,accuracy,ROC-AUC
0,Scaled RBF SVM,0.982,0.995
1,Scaled KNN,0.974,0.988
2,Gaussian Naive Bayes,0.939,0.988


## Intuition and trade-offs
- **SVM:** finds a decision boundary with a margin; kernels can model nonlinear boundaries. Scaling often matters a lot. Large datasets may make kernel SVM expensive.
- **KNN:** labels a point based on nearby examples. It is simple but prediction can be slow and performance depends on distance and scaling.
- **Naive Bayes:** combines probabilistic evidence using a conditional independence assumption. The assumption is often unrealistic, yet it can be a strong, fast baseline—especially for text with suitable variants.

## Practice
Change `n_neighbors` in KNN to 3, 7, and 15. Compare the validation score and explain why very small or very large values may fail.

In [2]:
for k in [3, 7, 15, 31]:
    est = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    p = est.predict_proba(X_test)[:,1]
    print("k=", k, "ROC-AUC=", round(roc_auc_score(y_test,p),3))

k= 3 ROC-AUC= 0.983
k= 7 ROC-AUC= 0.988
k= 15 ROC-AUC= 0.993
k= 31 ROC-AUC= 0.992
